<a href="https://colab.research.google.com/github/dssemugabi/AI-and-Nearal-networks/blob/main/tb_brain_desertation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
from PIL import Image
import numpy as np

def create_dummy_dataset(root_dir, num_samples_per_class=20):
    os.makedirs(root_dir, exist_ok=True)

    classes = ['no', 'yes']
    for cls_name in classes:
        class_dir = os.path.join(root_dir, cls_name)
        os.makedirs(class_dir, exist_ok=True)
        print(f"Created directory: {class_dir}")

        for i in range(num_samples_per_class):
            # Create a dummy image (e.g., 64x64 black image)
            dummy_image = Image.fromarray(np.zeros((64, 64, 3), dtype=np.uint8))
            image_path = os.path.join(class_dir, f"dummy_image_{i}.png")
            dummy_image.save(image_path)
            print(f"Created dummy image: {image_path}")

# Call the function to create the dummy dataset
dataset_root = "/tmp/brain-tumor-detection"
create_dummy_dataset(dataset_root, num_samples_per_class=20)
print(f"Dummy dataset created at {dataset_root}")


Created directory: /tmp/brain-tumor-detection/no
Created dummy image: /tmp/brain-tumor-detection/no/dummy_image_0.png
Created dummy image: /tmp/brain-tumor-detection/no/dummy_image_1.png
Created dummy image: /tmp/brain-tumor-detection/no/dummy_image_2.png
Created dummy image: /tmp/brain-tumor-detection/no/dummy_image_3.png
Created dummy image: /tmp/brain-tumor-detection/no/dummy_image_4.png
Created directory: /tmp/brain-tumor-detection/yes
Created dummy image: /tmp/brain-tumor-detection/yes/dummy_image_0.png
Created dummy image: /tmp/brain-tumor-detection/yes/dummy_image_1.png
Created dummy image: /tmp/brain-tumor-detection/yes/dummy_image_2.png
Created dummy image: /tmp/brain-tumor-detection/yes/dummy_image_3.png
Created dummy image: /tmp/brain-tumor-detection/yes/dummy_image_4.png
Dummy dataset created at /tmp/brain-tumor-detection


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# 1. VGG16 BACKBONE (split into first half and second half)
# =========================================================

class VGG16BackboneSplit(nn.Module):
    """
    VGG16-like feature extractor split into:
      - first_half: blocks 1 & 2
      - second_half: blocks 3, 4 & 5
    Input:  [B, 3, H, W]
    Mid:    [B, 128, H/4, W/4]
    Final:  [B, 512, H/32, W/32]
    """
    def __init__(self, in_channels=3):
        super().__init__()

        # Block 1: 64, 64 + pool (H/2, W/2)
        block1 = [
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]

        # Block 2: 128, 128 + pool (H/4, W/4)
        block2 = [
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]

        # Block 3: 256, 256, 256 + pool (H/8, W/8)
        block3 = [
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]

        # Block 4: 512, 512, 512 + pool (H/16, W/16)
        block4 = [
            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]

        # Block 5: 512, 512, 512 + pool (H/32, W/32)
        block5 = [
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]

        self.first_half = nn.Sequential(*block1, *block2)        # up to 128 channels
        self.second_half = nn.Sequential(*block3, *block4, *block5)

    def forward_first(self, x):
        # [B, 3, H, W] -> [B, 128, H/4, W/4]
        return self.first_half(x)

    def forward_second(self, x):
        # [B, C_mid, H/4, W/4] -> [B, 512, H/32, W/32]
        return self.second_half(x)


# =========================================================
# 2. TRANSFORMER ENCODERS (standard & group-query variant)
# =========================================================

class StandardTransformerBlock(nn.Module):
    """
    Classic Transformer encoder block (LayerNorm + MHA + MLP).
    Input:  [B, N, D]
    Output: [B, N, D]
    """
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Self-attention
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out

        # MLP
        h = self.norm2(x)
        x = x + self.mlp(h)
        return x


class GroupQueryTransformerBlock(nn.Module):
    """
    Transformer where queries are 'grouped' in the sequence dimension.
    Intuition: compress queries into groups, attend to full sequence, then
    broadcast back to original token positions.
    """
    def __init__(self, dim, num_heads, num_groups=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.num_groups = num_groups
        self.dim = dim

        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def _group_queries(self, x):
        """
        x: [B, N, D]
        Return grouped queries: [B, G, D], plus a list of group sizes.
        """
        B, N, D = x.shape
        G = min(self.num_groups, N)
        # split indices into G groups as evenly as possible
        sizes = []
        base = N // G
        rem = N % G
        start = 0
        groups = []
        for g in range(G):
            sz = base + (1 if g < rem else 0)
            end = start + sz
            groups.append(x[:, start:end, :])  # [B, sz, D]
            sizes.append(sz)
            start = end
        # mean over tokens inside each group -> [B, G, D]
        q_groups = torch.stack([grp.mean(dim=1) for grp in groups], dim=1)
        return q_groups, sizes

    def _broadcast_back(self, grouped_out, sizes):
        """
        grouped_out: [B, G, D]
        sizes: list of length G with token counts per group
        Return: [B, N, D] reconstructed by repeating group outputs.
        """
        B, G, D = grouped_out.shape
        outs = []
        for g in range(G):
            # expand group output to its original size
            o = grouped_out[:, g:g+1, :].expand(B, sizes[g], D)
            outs.append(o)
        return torch.cat(outs, dim=1)  # [B, N, D]

    def forward(self, x):
        """
        x: [B, N, D]
        """
        B, N, D = x.shape
        h = self.norm1(x)

        # group queries
        q_groups, sizes = self._group_queries(h)   # [B, G, D]
        # keys, values are full sequence
        k = h
        v = h
        # attention with grouped queries
        grouped_out, _ = self.attn(q_groups, k, v)  # [B, G, D]
        # broadcast back
        attn_per_token = self._broadcast_back(grouped_out, sizes)  # [B, N, D]

        x = x + attn_per_token

        h = self.norm2(x)
        x = x + self.mlp(h)
        return x


# =========================================================
# 3. MID-LEVEL "ViT" OPERATING ON CNN FEATURE MAPS
# =========================================================

class MidViTOnFeatureMap(nn.Module):
    """
    Take a feature map [B, C, H, W], interpret each spatial position as a token,
    apply one or more transformer blocks, and return the same shape.
    """
    def __init__(self, channels=128, depth=2, num_heads=4, grouped=False, num_groups=4, dropout=0.0):
        super().__init__()
        blocks = []
        for _ in range(depth):
            if grouped:
                blocks.append(GroupQueryTransformerBlock(channels, num_heads, num_groups, dropout=dropout))
            else:
                blocks.append(StandardTransformerBlock(channels, num_heads, dropout=dropout))
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape
        # [B, C, H, W] -> [B, H*W, C]
        x_seq = x.view(B, C, H * W).transpose(1, 2)

        for blk in self.blocks:
            x_seq = blk(x_seq)  # [B, N, C]

        # back to [B, C, H, W]
        x_out = x_seq.transpose(1, 2).view(B, C, H, W)
        return x_out


# =========================================================
# 4. FUSION ViT (shares information between branches)
# =========================================================

class FusionViT(nn.Module):
    """
    Tiny ViT to share information between final features of both branches.
    Input: 2 tokens per sample (v1, v2), shape [B, 2, D]
    Output: [B, 2, D]
    """
    def __init__(self, dim=512, depth=1, num_heads=4, dropout=0.0):
        super().__init__()
        self.blocks = nn.ModuleList([
            StandardTransformerBlock(dim, num_heads, dropout=dropout)
        for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, 2, D]
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x  # [B, 2, D]


# =========================================================
# 5. FULL MODEL: TWO BRANCHES WITH VGG16 + MID ViTs + FUSION ViT
# =========================================================

class TwoBranchVGG16WithViTs(nn.Module):
    """
    Branch 1:
      x1 -> first_half VGG16 -> MidViT (standard) -> second_half VGG16 -> GAP -> v1

    Branch 2:
      x2 -> first_half VGG16 -> MidViT (group query) -> second_half VGG16 -> GAP -> v2

    Fusion:
      stack [v1, v2] -> FusionViT -> flatten -> MLP head -> 1 logit (binary)
    """
    def __init__(self,
                 in_channels=3,
                 mid_channels=128,   # from VGG16 first_half output
                 final_channels=512, # from VGG16 second_half output
                 mid_depth_standard=2,
                 mid_depth_grouped=2,
                 mid_heads=4,
                 mid_groups=4,
                 fusion_dim=512,
                 fusion_depth=1,
                 fusion_heads=4):
        super().__init__()

        # Branch 1: its own VGG16 split
        self.vgg1 = VGG16BackboneSplit(in_channels=in_channels)
        # Branch 2: its own VGG16 split (could share weights, but you didn't say so)
        self.vgg2 = VGG16BackboneSplit(in_channels=in_channels)

        # Mid ViTs on feature maps
        self.mid_vit1 = MidViTOnFeatureMap(
            channels=mid_channels,
            depth=mid_depth_standard,
            num_heads=mid_heads,
            grouped=False
        )

        self.mid_vit2 = MidViTOnFeatureMap(
            channels=mid_channels,
            depth=mid_depth_grouped,
            num_heads=mid_heads,
            grouped=True,
            num_groups=mid_groups
        )

        # Fusion ViT on final pooled features
        self.fusion_vit = FusionViT(dim=fusion_dim, depth=fusion_depth, num_heads=fusion_heads)

        # Final MLP head for binary classification (1 logit)
        self.head = nn.Sequential(
            nn.Linear(fusion_dim * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 1)  # binary classification logit
        )

    def _branch_forward(self, x, vgg_split, mid_vit):
        # x: [B, 3, H, W]
        # 1) first half of VGG16
        f_mid = vgg_split.forward_first(x)  # [B, 128, H/4, W/4]

        # 2) mid ViT on feature map
        f_mid = mid_vit(f_mid)             # still [B, 128, H/4, W/4]

        # 3) rest of VGG16
        f_final = vgg_split.forward_second(f_mid)  # [B, 512, H/32, W/32]

        # 4) global average pooling to get vector
        v = F.adaptive_avg_pool2d(f_final, 1).view(x.size(0), -1)  # [B, 512]
        return v

    def forward(self, x1, x2):
        """
        x1, x2: [B, 3, H, W]
        """
        # Branch 1
        v1 = self._branch_forward(x1, self.vgg1, self.mid_vit1)  # [B, 512]
        # Branch 2
        v2 = self._branch_forward(x2, self.vgg2, self.mid_vit2)  # [B, 512]

        # Stack into 2-token sequence
        fused_tokens = torch.stack([v1, v2], dim=1)  # [B, 2, 512]

        # Share info via Fusion ViT
        fused_tokens = self.fusion_vit(fused_tokens)  # [B, 2, 512]

        # Flatten both tokens
        fused_vec = fused_tokens.view(x1.size(0), -1)  # [B, 1024]

        # Binary logit
        logit = self.head(fused_vec)  # [B, 1]
        return logit


# =========================================================
# 6. USAGE EXAMPLE
# =========================================================

if __name__ == "__main__":
    model = TwoBranchVGG16WithViTs()

    # Example batch of two inputs
    x1 = torch.randn(4, 3, 256, 256)  # [B, C, H, W]
    x2 = torch.randn(4, 3, 256, 256)

    logits = model(x1, x2)
    print("logits shape:", logits.shape)  # [4, 1]

    # Example loss for binary classification
    labels = torch.randint(0, 2, (4, 1)).float()
    criterion = nn.BCEWithLogitsLoss()
    loss = criterion(logits, labels)
    loss.backward()
    print("loss:", loss.item())


logits shape: torch.Size([4, 1])
loss: 0.6854233145713806


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
import torchvision.transforms as T
import torch

# ============================================================
# BINARY LABEL MAP (YES / NO)
# ============================================================

label_map = {
    "no": 0,
    "yes": 1
}

classes = list(label_map.keys())


# ============================================================
# DATASET CLASS — TWO AUGMENTED VIEWS
# ============================================================

class YesNoDataset(Dataset):
    def __init__(self, samples, transform1=None, transform2=None):
        self.samples = samples
        self.transform1 = transform1
        self.transform2 = transform2 if transform2 else transform1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        x1 = self.transform1(img)
        x2 = self.transform2(img)

        return x1, x2, label


# ============================================================
# LOAD ALL IMAGES (YES / NO)
# ============================================================

def load_samples(root):
    samples = []

    for cls in classes:
        folder = os.path.join(root, cls)
        for f in os.listdir(folder):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                samples.append((os.path.join(folder, f), label_map[cls]))

    return samples


# ============================================================
# FINAL 70 / 10 / 20 STRATIFIED DATA LOADERS
# ============================================================

def get_yesno_loaders(root, batch_size=64, seed=42):

    # ---------------------------
    # TRAIN AUGMENTATION
    # ---------------------------
    train_tfm = T.Compose([
        T.Resize((128, 128)),
        T.RandomHorizontalFlip(0.5),
        T.RandomRotation(15),
        T.ColorJitter(0.1, 0.1),
        T.RandomAffine(10, translate=(0.05, 0.05)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406],
                    [0.229, 0.224, 0.225])
    ])

    # ---------------------------
    # VAL/TEST TRANSFORM
    # ---------------------------
    test_tfm = T.Compose([
        T.Resize((128, 128)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406],
                    [0.229, 0.224, 0.225])
    ])

    # -------------------------------------------------
       # LOAD SAMPLES
    # -------------------------------------------------
    all_samples = load_samples(root)
    paths  = [s[0] for s in all_samples]
    labels = [s[1] for s in all_samples]

    # -------------------------------------------------
    # FIRST SPLIT: TRAIN (70%) / TEMP (30%)
    # -------------------------------------------------
    sss1 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.30,      # 30% goes to temp (val+test)
        random_state=seed
    )
    train_idx, temp_idx = next(sss1.split(paths, labels))

    train_samples = [all_samples[i] for i in train_idx]
    temp_samples  = [all_samples[i] for i in temp_idx]

    # -------------------------------------------------
    # SECOND SPLIT: TEMP → VAL (10%) + TEST (20%)
    # temp = 30% → val should be 10% and test 20%
    # so inside temp: val = 1/3, test = 2/3
    # -------------------------------------------------
    temp_paths  = [s[0] for s in temp_samples]
    temp_labels = [s[1] for s in temp_samples]

    sss2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=2/3,       # 2/3 of temp = 20% total (test)
        random_state=seed
    )
    val_idx, test_idx = next(sss2.split(temp_paths, temp_labels))

    # NOTE: train_idx from sss2 → val, test_idx → test
    val_samples  = [temp_samples[i] for i in val_idx]
    test_samples = [temp_samples[i] for i in test_idx]

    # -------------------------
    # SHOW EXACT COUNTS
    # -------------------------
    print("\nYES/NO Dataset Split (70/10/20):")
    print("Train =", len(train_samples))
    print("Val   =", len(val_samples))
    print("Test  =", len(test_samples))

    # -------------------------
    # DATASETS
    # -------------------------
    train_ds = YesNoDataset(train_samples, train_tfm, train_tfm)
    val_ds   = YesNoDataset(val_samples, test_tfm, test_tfm)
    test_ds  = YesNoDataset(test_samples, test_tfm, test_tfm)

    # -------------------------
    # DATALOADERS
    # -------------------------
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float("inf")

    hist_train_loss = []
    hist_val_loss = []
    hist_val_acc = []

    for epoch in range(1, epochs + 1):

        # -----------------------
        # TRAIN
        # -----------------------
        model.train()
        train_loss = 0.0

        for img1, img2, labels in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
            img1 = img1.to(device)
            img2 = img2.to(device)
            labels = labels.float().unsqueeze(1).to(device)   # [B,1]

            optimizer.zero_grad()

            logits = model(img1, img2)    # two-branch model
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * img1.size(0)

        train_loss /= len(train_loader.dataset)

        # -----------------------
        # VALIDATION
        # -----------------------
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for img1, img2, labels in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
                img1 = img1.to(device)
                img2 = img2.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                logits = model(img1, img2)
                loss = criterion(logits, labels)

                val_loss += loss.item() * img1.size(0)

                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()

                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(val_loader.dataset)
        val_acc = correct / total if total > 0 else 0.0

        hist_train_loss.append(train_loss)
        hist_val_loss.append(val_loss)
        hist_val_acc.append(val_acc)

        print(f"\nEpoch {epoch}: "
              f"Train Loss = {train_loss:.4f} | "
              f"Val Loss = {val_loss:.4f} | "
              f"Val Acc = {val_acc:.4f}")

        # Save best model (lowest val loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("🔥 Saved new best model (lowest val loss).")

    print("\n✅ Training complete. Best model saved as 'best_model.pth'.")

def test_model(model, test_loader, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    model = model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for img1, img2, labels in tqdm(test_loader, desc="Testing"):
            img1 = img1.to(device)
            img2 = img2.to(device)
            labels = labels.float().unsqueeze(1).to(device)   # [B,1]

            logits = model(img1, img2)          # two-branch forward
            probs = torch.sigmoid(logits)       # convert logits → probabilities
            preds = (probs > 0.5).float()       # threshold = 0.5

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total if total > 0 else 0.0
    print(f"\n🎯 TEST ACCURACY: {test_acc:.4f}")


from tqdm import tqdm

if __name__ == "__main__":
    # Root folder where your data lives
    # Example: "/mnt/data/dataset" in our environment
    root = "/tmp/brain-tumor-detection"

    train_loader, val_loader, test_loader = get_yesno_loaders(
        root=root,
        batch_size=16,
        seed=42
    )

    # 2. Create model (from previous definition)
    model = TwoBranchVGG16WithViTs()

    # 3. Train (with validation + best model saving)
    train_model(model, train_loader, val_loader, epochs=50, lr=0.0001)

    # 4. Test best model
    test_model(model, test_loader)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# 1. GET RAW PROBABILITIES FROM MODEL
# ============================================================
def get_test_probs(model, test_loader, device=None):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    model = model.to(device)
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for img1, img2, labels in tqdm(test_loader, desc="Getting predictions"):
            img1 = img1.to(device)
            img2 = img2.to(device)

            logits = model(img1, img2)                  # [B,1]
            probs = torch.sigmoid(logits).cpu().numpy().flatten()

            all_probs.extend(probs)
            all_labels.extend(labels.numpy().flatten())  # already int 0/1

    return np.array(all_labels), np.array(all_probs)


# ============================================================
# 2. SAVE CONFUSION MATRIX IMAGE
# ============================================================
def save_confusion_matrix(y_true, y_pred, filename):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Normal", "Tuberculosis"],
        yticklabels=["Normal", "Tuberculosis"]
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

    print(f"📁 Confusion matrix saved to: {filename}")


# ============================================================
# 3. FULL TEST EVALUATION WITH METRICS
# ============================================================
def test_model_full(model, test_loader, threshold=0.5):

    print("\n=== Running Full Test Evaluation ===")

    # 1) get raw data
    y_true, y_probs = get_test_probs(model, test_loader)

    # 2) apply threshold
    y_pred = (y_probs > threshold).astype(int)

    # 3) calculate metrics
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1  = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_probs)
    except:
        auc = None

    # 4) print all results
    print("\n===== TEST RESULTS =====")
    print(f"🎯 Accuracy:  {acc:.4f}")
    print(f"🔍 Precision: {pre:.4f}")
    print(f"📈 Recall:    {rec:.4f}")
    print(f"🏅 F1-Score:  {f1:.4f}")
    if auc is not None:
        print(f"💠 AUC-ROC:   {auc:.4f}")
    else:
        print("💠 AUC-ROC:   Cannot compute (only one class present).")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # 5) save confusion matrix
    save_confusion_matrix(y_true, y_pred, "tb_confusion_matrix.png")

    return y_true, y_pred, y_probs

In [ ]:
y_true, y_pred, y_probs = test_model_full(model, test_loader, threshold=0.5)

In [ ]:
import os

print(os.listdir("/kaggle/working"))


In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/best_model.pth')
FileLink('/kaggle/working/tb_confusion_matrix.png')


In [ ]:
def save_and_show_confusion_matrix(y_true, y_pred, filename):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Normal", "Tuberculosis"],
        yticklabels=["Normal", "Tuberculosis"]
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()

    # Save the image
    plt.savefig(filename, dpi=300)

    # Show it inside the notebook
    plt.show()

    print(f"📁 Confusion matrix saved to: {filename}")

save_and_show_confusion_matrix(y_true, y_pred, "tb_confusion_matrix.png")
